In [3]:
!pip install streamlit
import streamlit as st
import numpy as np
import cv2
from PIL import Image
import os
import time
import random
import sqlite3
import json
from datetime import datetime

try:
    from fpdf import FPDF
    PDF_AVAILABLE = True
except ImportError:
    PDF_AVAILABLE = False

MODEL LOADER

In [5]:
import threading
_model_cache = {}
_model_lock  = threading.Lock()

def load_tb_model():
    """
    Load tb_classifier.pt via Ultralytics YOLO.
    Returns the model on success, None if the file is a dummy or
    ultralytics / torch are not installed.
    Caches the result so the model is only loaded once per session.
    """
    with _model_lock:
        if "tb" in _model_cache:
            return _model_cache["tb"]
        model_path = "models/tb_classifier.pt"
        try:
            # Sanity-check: dummy placeholder is tiny text file
            with open(model_path, "rb") as f:
                header = f.read(16)
            if b"Dummy" in header or b"dummy" in header:
                _model_cache["tb"] = None
                return None
            from ultralytics import YOLO
            model = YOLO(model_path)
            _model_cache["tb"] = model
            return model
        except Exception:
            _model_cache["tb"] = None
            return None


CONFIGURE

In [6]:
DB_PATH = "pink_edge_cache.db"

BI_RADS_OPTIONS = [
    "BI-RADS 0 - Incomplete (Additional imaging needed)",
    "BI-RADS 1 - Negative",
    "BI-RADS 2 - Benign finding",
    "BI-RADS 3 - Probably benign (6-month follow-up)",
    "BI-RADS 4A - Low suspicion (Biopsy recommended)",
    "BI-RADS 4B - Moderate suspicion",
    "BI-RADS 4C - High suspicion",
    "BI-RADS 5 - Highly suggestive of malignancy",
    "BI-RADS 6 - Known biopsy-proven malignancy",
]

ACR_DENSITY_OPTIONS = [
    "A - Almost entirely fatty",
    "B - Scattered fibroglandular density",
    "C - Heterogeneously dense",
    "D - Extremely dense",
]

MODELS = [
    "Mammography (YOLOv8-OBB)",
    "Tuberculosis (Chest X-Ray)",
    "Maternal Health (Ultrasound)",
]

URDU TRANSLATION

In [7]:
TR = {
    "Pink Edge AI": "پنک ایج اے آئی",
    "Clinical Intelligence Platform": "کلیںیکل انٹیلیجنس پلیٹ فارم",
    "Offline Edge AI Triage": "آف لائن ایج اے آئی تشخیص",
    "Edge AI Medical Analysis on Rockchip RK3588 NPU": "راک چیپ RK3588 NPU پر ایج اے آئی طبی تجزیہ",
    "Dashboard": "ڈیش بورڈ",
    "Hospital Hub": "ہسپتال مرکز",
    "Cloud Sync": "کلاؤڈ سنک",
    "Fully Offline": "مکمل آف لائن",
    "GSM Failover": "جی ایس ایم فیل اوور",
    "English": "انگریزی",
    "اردو": "اردو",
    "Select AI Model": "اے آئی ماڈل منتخب کریں",
    "Upload Patient Scan": "مریض کا اسکین اپ لوڈ کریں",
    "Run Triage": "تشخیص چلائیں",
    "Ingest DICOM": "ڈائیکوم داخل کریں",
    "Save to Cache": "کیش میں محفوظ کریں",
    "Sync to Cloud": "کلاؤڈ میں سنک کریں",
    "Check OTA": "او ٹی اے چیک کریں",
    "Reset Session": "سیشن ری سیٹ کریں",
    "Show Detection Overlay": "کھوج اوورلے دکھائیں",
    "Network Mode": "نیٹ ورک موڈ",
    "Language": "زبان",
    "Medical Image Analysis": "طبی تصویر تجزیہ",
    "DICOM Metadata": "ڈائیکوم میٹا ڈیٹا",
    "AI Triage Verdict": "اے آئی تشخیص فیصلہ",
    "Awaiting Analysis": "تجزیہ کا انتظار",
    "Run triage to begin": "شروع کرنے کے لیے تشخیص چلائیں",
    "Telemetry Log": "ٹیلی میٹری لاگ",
    "BI-RADS Assessment": "بی آر اے ڈی ایس تشخیص",
    "ACR Breast Density": "اے سی آر چھاتی کثافت",
    "Confirm Assessment": "تصدیق تشخیص",
    "Download Text Report": "متن رپورٹ ڈاؤن لوڈ کریں",
    "Download PDF Report": "پی ڈی ایف رپورٹ ڈاؤن لوڈ کریں",
    "Report cached to offline hardware node": "رپورٹ لوکل ہارڈویئر نوڈ پر محفوظ ہو گئی",
    "Sync complete": "سنک مکمل",
    "Cached Reports": "کیش شدہ رپورٹس",
    "Unsynced": "غیر سنک شدہ",
    "Synced": "سنک شدہ",
    "IoT Queue": "آئی او ٹی قطار",
    "No alerts pending": "کوئی الرٹ زیر التوا نہیں",
    "Total Alerts": "کل الرٹس",
    "Critical Cases": "تنقی کیسز",
    "Patient ID": "مریض آئی ڈی",
    "Modality": "ماڈیلٹی",
    "Verdict": "فیصلہ",
    "Timestamp": "ٹائم اسٹیمپ",
    "No reports cached": "کوئی رپورٹ کیش نہیں",
    "Alibaba Cloud Integration": "علی بابا کلاؤڈ انٹیگریشن",
    "Hybrid-Edge Architecture": "ہائبرڈ ایج آرکیٹیکچر",
    "Local Cache Status": "لوکل کیش حالت",
    "Confidence": "اعتماد",
    "Latency": "تاخیر",
    "Localization": "مقام",
    "Classification": "درجہ بندی",
    "Inference Time": "انفیرنس وقت",
    "Hardware": "ہارڈویئر",
    "NPU Load": "این پی یو لوڈ",
    "Power": "پاور",
    "Temperature": "درجہ حرارت",
    "Active Model": "فعال ماڈل",
    "Immediate Action Required": "فوری اقدام ضروری",
    "Patient should be referred for specialist consultation": "مریض کو ماہر مشاورت کے لیے بھیجا جائے",
    "Network Status": "نیٹ ورک حالت",
    "Alert Stream": "الرٹ اسٹریم",
    "Stats": "شماریات",
    "Navigate": "نیویگیٹ",
    "Actions": "اقدامات",
    "Hardware Diagnostics": "ہارڈویئر تشخیص",
    "Report safely cached to offline hardware node.": "رپورٹ لوکل ہارڈویئر نوڈ پر محفوظ کر دی گئی۔",
    "Sync complete! Reports uploaded to Alibaba Cloud.": "سنک مکمل! رپورٹس علی بابا کلاؤڈ پر اپ لوڈ ہو گئیں۔",
    "Allied Hospital Faisalabad": "اتحادی ہسپتال فیصل آباد",
    "Urban Receiving Terminal": "شہری وصول ٹرمینل",
    "2G GSM Critical Alert Monitor": "2G جی ایس ایم تنقی الرٹ مانیٹر",
    "No OTA updates available.": "کوئی او ٹی اے اپڈیٹس دستیاب نہیں۔",
    "No reports cached yet.": "ابھی تک کوئی رپورٹ کیش نہیں۔",
    "DICOM scan ingested successfully!": "ڈائیکوم اسکین کامیابی سے داخل ہو گیا!",
    "Routine follow-up recommended": "روایتی پیروی کی سفارش کی گئی",
    "No immediate action required": "کوئی فوری اقدام ضروری نہیں",
    "Low risk - routine screening": "کم خطرہ - روایتی اسکریننگ",
}


def t(text):
    if st.session_state.get("urdu_mode", False):
        return TR.get(text, text)
    return text


def safe_pdf_text(text):
    if text is None:
        return ""
    text = str(text)
    replacements = {"—": "-", "–": "-", "’": "'", """: '"', """: '"', "•": "-", "→": "->"}
    for uni, asc in replacements.items():
        text = text.replace(uni, asc)
    return text.encode('latin-1', 'replace').decode('latin-1')

CLINICAL DATA GENERATORS — Realistic & Varied

In [8]:

def generate_new_patient():
    """Generate a completely new unique patient each time."""
    return {
        "pat_id": random.randint(10000000, 99999999),
        "pat_age": random.randint(28, 75),
    }


def generate_mammography_result():
    """Generate varied mammography results — not always BI-RADS 5."""
    scenarios = [
        {"bi_rads": BI_RADS_OPTIONS[1], "acr": ACR_DENSITY_OPTIONS[0], "verdict": "Normal",
         "sub": "No Anomalies Detected", "css": "success", "loc": "No focal lesion identified",
         "extra": "ACR Class A", "vicon": "✅", "confidence": random.uniform(95.0, 99.2),
         "sms": "BI-RADS:1", "is_critical": False},
        {"bi_rads": BI_RADS_OPTIONS[2], "acr": ACR_DENSITY_OPTIONS[1], "verdict": "Benign Finding",
         "sub": "Simple cyst identified", "css": "success", "loc": "Upper Outer Quadrant",
         "extra": "ACR Class B", "vicon": "✅", "confidence": random.uniform(92.0, 96.5),
         "sms": "BI-RADS:2", "is_critical": False},
        {"bi_rads": BI_RADS_OPTIONS[3], "acr": ACR_DENSITY_OPTIONS[1], "verdict": "Probably Benign",
         "sub": "Likely fibroadenoma - 6-month follow-up", "css": "success", "loc": "Lower Inner Quadrant",
         "extra": "ACR Class B", "vicon": "✅", "confidence": random.uniform(88.0, 93.0),
         "sms": "BI-RADS:3", "is_critical": False},
        {"bi_rads": BI_RADS_OPTIONS[4], "acr": ACR_DENSITY_OPTIONS[2], "verdict": "Low Suspicion",
         "sub": "Suspicious morphology - Biopsy recommended", "css": "danger", "loc": "Upper Outer Quadrant",
         "extra": "ACR Class C", "vicon": "⚠️", "confidence": random.uniform(82.0, 88.0),
         "sms": "BI-RADS:4A", "is_critical": True},
        {"bi_rads": BI_RADS_OPTIONS[5], "acr": ACR_DENSITY_OPTIONS[2], "verdict": "Moderate Suspicion",
         "sub": "Irregular mass - Biopsy advised", "css": "danger", "loc": "Central Region",
         "extra": "ACR Class C", "vicon": "⚠️", "confidence": random.uniform(85.0, 90.0),
         "sms": "BI-RADS:4B", "is_critical": True},
        {"bi_rads": BI_RADS_OPTIONS[7], "acr": ACR_DENSITY_OPTIONS[2], "verdict": "BI-RADS 5",
         "sub": "Highly Suspicious - Malignancy likely", "css": "danger", "loc": "Upper Outer Quadrant",
         "extra": "ACR Class C", "vicon": "⚠️", "confidence": random.uniform(91.0, 96.0),
         "sms": "BI-RADS:5", "is_critical": True},
        {"bi_rads": BI_RADS_OPTIONS[7], "acr": ACR_DENSITY_OPTIONS[3], "verdict": "BI-RADS 5",
         "sub": "Highly Suspicious - Spiculated mass", "css": "danger", "loc": "Upper Outer Quadrant",
         "extra": "ACR Class D", "vicon": "⚠️", "confidence": random.uniform(93.0, 97.5),
         "sms": "BI-RADS:5", "is_critical": True},
    ]
    return random.choice(scenarios)


TB SEVERITY SCORING (replaces BI-RADS for chest X-ray)
# Adapted from WHO TB Severity Index & Chest X-Ray scoring

In [9]:
TB_SEVERITY_LEVELS = [
    "S0 - No active disease",
    "S1 - Minimal (unilateral, no cavity)",
    "S2 - Moderate (bilateral / cavity < 2 cm)",
    "S3 - Advanced (large cavity / miliary pattern)",
]

TB_LUNG_ZONES = [
    "Upper Zone",
    "Middle Zone",
    "Lower Zone",
    "Bilateral",
]

def _run_tb_inference(image_np):
    """
    Attempt real YOLOv8 inference on a numpy uint8 image.
    Returns (class_id, confidence, box_xyxy) or None on failure.
    class_id 0 = TB Negative, 1 = TB Positive (adjust to your label map).
    """
    model = load_tb_model()
    if model is None:
        return None
    try:
        results = model.predict(source=image_np, imgsz=512, conf=0.25,
                                verbose=False, device="cpu")
        if not results or len(results[0].boxes) == 0:
            # No detection → model says clear
            return {"class_id": 0, "confidence": random.uniform(94.0, 98.5),
                    "box": None}
        boxes  = results[0].boxes
        # Pick highest-confidence detection
        best   = int(boxes.conf.argmax())
        cls_id = int(boxes.cls[best].item())
        conf   = float(boxes.conf[best].item()) * 100.0
        box    = boxes.xyxy[best].tolist()  # [x1,y1,x2,y2]
        return {"class_id": cls_id, "confidence": conf, "box": box}
    except Exception:
        return None


def generate_tb_result(image_np=None):
    """
    Generate TB triage result.
    If a real model is loaded and image_np is provided, runs actual inference.
    Falls back to simulation when model is a dummy or unavailable.
    """
    real = _run_tb_inference(image_np) if image_np is not None else None

    if real is not None:
        # ---- REAL INFERENCE PATH ----
        is_positive = (real["class_id"] == 1)
        conf        = real["confidence"]
        if is_positive:
            # Map confidence to severity heuristic
            if conf >= 90:
                severity = TB_SEVERITY_LEVELS[3]
                sub      = "Advanced bilateral disease with large cavity"
                loc      = "Bilateral upper lobes"
                extra    = "Large cavity / miliary pattern"
            elif conf >= 75:
                severity = TB_SEVERITY_LEVELS[2]
                sub      = "Moderate disease with cavitation detected"
                loc      = "Right Upper Lobe"
                extra    = "Cavity < 2 cm"
            else:
                severity = TB_SEVERITY_LEVELS[1]
                sub      = "Minimal lesion — unilateral, no cavitation"
                loc      = "Right Upper Zone"
                extra    = "No cavity formation"
            return {
                "verdict": "TB Positive",
                "sub": sub,
                "css": "danger",
                "loc": loc,
                "extra": extra,
                "tb_severity": severity,
                "vicon": "⚠️",
                "confidence": conf,
                "sms": f"TB:POS|SEV:{severity[1]}",
                "is_critical": True,
                "real_inference": True,
                "detection_box": real.get("box"),
                # Keep bi_rads/acr keys for cache schema compatibility
                "bi_rads": "N/A (TB Triage)",
                "acr": "N/A",
            }
        else:
            return {
                "verdict": "TB Negative",
                "sub": "No active lesions detected by classifier",
                "css": "success",
                "loc": "Lungs clear",
                "extra": "No cavity formation",
                "tb_severity": TB_SEVERITY_LEVELS[0],
                "vicon": "✅",
                "confidence": conf,
                "sms": "TB:NEG",
                "is_critical": False,
                "real_inference": True,
                "detection_box": None,
                "bi_rads": "N/A (TB Triage)",
                "acr": "N/A",
            }


SIMULATION FALLBACK

In [12]:
def generate_tb_simulation_result():
    scenarios = [
            {"verdict": "TB Negative", "sub": "No active lesions detected", "css": "success",
             "loc": "Lungs clear", "extra": "No cavity formation",
             "tb_severity": TB_SEVERITY_LEVELS[0],
             "vicon": "✅", "confidence": random.uniform(94.0, 98.5),
             "sms": "TB:NEG", "is_critical": False, "real_inference": False,
             "detection_box": None, "bi_rads": "N/A (TB Triage)", "acr": "N/A"},
            {"verdict": "TB Negative", "sub": "Old calcified granuloma — inactive", "css": "success",
             "loc": "Right Upper Lobe", "extra": "No active disease",
             "tb_severity": TB_SEVERITY_LEVELS[0],
             "vicon": "✅", "confidence": random.uniform(90.0, 95.0),
             "sms": "TB:NEG", "is_critical": False, "real_inference": False,
             "detection_box": None, "bi_rads": "N/A (TB Triage)", "acr": "N/A"},
            {"verdict": "TB Positive", "sub": "Active lesion detected — cavity formation", "css": "danger",
             "loc": "Right Upper Lobe", "extra": "Cavity Formation",
             "tb_severity": TB_SEVERITY_LEVELS[2],
             "vicon": "⚠️", "confidence": random.uniform(85.0, 92.0),
             "sms": "TB:POS|SEV:2", "is_critical": True, "real_inference": False,
             "detection_box": None, "bi_rads": "N/A (TB Triage)", "acr": "N/A"},
            {"verdict": "TB Positive", "sub": "Bilateral infiltrates — active disease", "css": "danger",
             "loc": "Bilateral Upper Lobes", "extra": "Multi-focal involvement",
             "tb_severity": TB_SEVERITY_LEVELS[3],
             "vicon": "⚠️", "confidence": random.uniform(87.0, 93.0),
             "sms": "TB:POS|SEV:3", "is_critical": True, "real_inference": False,
             "detection_box": None, "bi_rads": "N/A (TB Triage)", "acr": "N/A"},
        ]
    return random.choice(scenarios)

def generate_fetal_result():
    """Generate varied fetal health results."""
    scenarios = [
        {"bi_rads": BI_RADS_OPTIONS[1], "acr": ACR_DENSITY_OPTIONS[0], "verdict": "Fetal Health Normal",
         "sub": "No anomalies detected", "css": "success", "loc": "Intrauterine",
         "extra": f"Gestational Age: {random.randint(18, 36)}W", "vicon": "✅",
         "confidence": random.uniform(96.0, 99.0), "sms": "FH:OK", "is_critical": False},
        {"bi_rads": BI_RADS_OPTIONS[1], "acr": ACR_DENSITY_OPTIONS[0], "verdict": "Fetal Health Normal",
         "sub": "Normal cardiac activity", "css": "success", "loc": "Intrauterine",
         "extra": f"Gestational Age: {random.randint(20, 38)}W", "vicon": "✅",
         "confidence": random.uniform(95.0, 98.5), "sms": "FH:OK", "is_critical": False},
    ]
    return random.choice(scenarios)


def generate_hw_stats():
    """Generate fresh hardware stats each triage."""
    return {
        "load": f"{random.randint(30, 45)}%",
        "power": f"{random.uniform(5.8, 6.5):.1f}W",
        "temp": f"{random.randint(38, 45)}C",
    }


def generate_net_stats():
    """Generate fresh network stats."""
    return {
        "signal": f"-{random.randint(75, 95)} dBm",
        "bhus": random.randint(10, 20),
        "module": "ONLINE" if random.random() > 0.1 else "UNSTABLE",
    }

DATABASE

In [13]:
def init_db():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""CREATE TABLE IF NOT EXISTS cached_reports (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        patient_id TEXT, patient_age INTEGER, modality TEXT,
        model_used TEXT, bi_rads TEXT, acr_density TEXT,
        verdict TEXT, localization TEXT, confidence REAL,
        inference_time REAL, timestamp TEXT, synced INTEGER DEFAULT 0,
        report_json TEXT
    )""")
    conn.commit()
    conn.close()


def save_to_cache(data):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("""INSERT INTO cached_reports
        (patient_id, patient_age, modality, model_used, bi_rads, acr_density,
         verdict, localization, confidence, inference_time, timestamp, synced, report_json)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)""",
        (data["patient_id"], data["patient_age"], data["modality"], data["model_used"],
         data["bi_rads"], data["acr_density"], data["verdict"], data["localization"],
         data["confidence"], data["inference_time"], data["timestamp"], 0,
         json.dumps(data, default=str)))
    conn.commit()
    conn.close()


def get_cached_reports():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT * FROM cached_reports ORDER BY id DESC")
    rows = c.fetchall()
    conn.close()
    return rows


def get_unsynced_reports():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT * FROM cached_reports WHERE synced = 0")
    rows = c.fetchall()
    conn.close()
    return rows


def mark_as_synced(report_ids):
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    for rid in report_ids:
        c.execute("UPDATE cached_reports SET synced = 1 WHERE id = ?", (rid,))
    conn.commit()
    conn.close()


def get_cache_count():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM cached_reports")
    n = c.fetchone()[0]
    conn.close()
    return n


def get_unsynced_count():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM cached_reports WHERE synced = 0")
    n = c.fetchone()[0]
    conn.close()
    return n

ALIBABA CLOUD SIMULATION

In [14]:
def simulate_iot_sync(reports):
    results = []
    for r in reports:
        results.append({
            "report_id": r[0], "patient_id": r[1],
            "payload": f"ID:{r[1]}|BR:{r[5]}|TS:{r[11]}",
            "iot_id": f"IOT-{random.randint(100000,999999)}",
        })
        time.sleep(0.03)
    return results


def simulate_oss_upload(report):
    bi_rads = report[5] or ""
    high = "4" in bi_rads or "5" in bi_rads
    if high:
        return {"status": "UPLOADED", "key": f"pink-edge/hr/{report[1]}.jpg", "kb": random.randint(45,85), "high": True}
    return {"status": "SKIPPED", "key": None, "kb": 0, "high": False}


def simulate_acr_check():
    return {"current": "v2.1.0", "available": "v2.2.1", "size": 14.2,
            "registry": "registry.ap-south-1.aliyuncs.com/pink-edge/ai-models"}


REPORT GENERATION

In [15]:
def generate_text_report(d):
    is_tb = "Tuberculosis" in d.get("model_used", "") or d.get("modality", "") == "DX"
    r = f"""
PINK EDGE AI - CLINICAL DIAGNOSTIC REPORT
==========================================

PATIENT: {d['patient_id']}    AGE: {d['patient_age']}    DATE: {d['timestamp']}
MODALITY: {d['modality']}      INSTITUTION: Rural BHU Faisalabad

AI ANALYSIS
-----------
Model: {d['model_used']}
Verdict: {d['verdict']}
Confidence: {d['confidence']:.1f}%
Inference: {d['inference_time']}s on RK3588 NPU (INT8)
Localization: {d['localization']}

CLINICAL ASSESSMENT
-------------------
"""
    if is_tb:
        r += f"""TB Severity (WHO Index): {d['bi_rads']}
Lung Zone: {d['acr_density']}
"""
        if "Positive" in d.get("verdict", ""):
            r += "\nURGENT: Patient must be isolated and referred to district TB centre.\n"
            r += "Notify LHW supervisor. Sputum GeneXpert test required.\n"
        else:
            r += "\nNo active TB detected. Routine follow-up per BHU protocol.\n"
    else:
        r += f"""BI-RADS: {d['bi_rads']}
ACR Density: {d['acr_density']}
"""
        if "5" in d.get("bi_rads","") or "4C" in d.get("bi_rads",""):
            r += "\nURGENT: Immediate oncology referral. Biopsy recommended.\n"
        elif "4A" in d.get("bi_rads","") or "4B" in d.get("bi_rads",""):
            r += "\nReferral recommended. Biopsy evaluation advised.\n"
        elif "3" in d.get("bi_rads",""):
            r += "\n6-month follow-up imaging recommended.\n"
        else:
            r += "\nRoutine screening per guidelines.\n"
    r += f"""
NETWORK: {d.get('network_mode','Offline')}
SYNC: {'Pending' if d.get('synced')==0 else 'Complete'}

Generated by Pink Edge AI v5.3 - Alibaba Cloud AI Hackathon 2026
"""
    return r


def generate_pdf_report(d):
    if not PDF_AVAILABLE:
        return None
    try:
        pdf = FPDF(orientation="P", unit="mm", format="A4")
        pdf.add_page()

        pdf.set_fill_color(13, 148, 136)
        pdf.rect(0, 0, 210, 45, "F")
        pdf.set_text_color(255, 255, 255)
        pdf.set_font("Helvetica", "B", 20)
        pdf.cell(0, 18, "PINK EDGE AI", ln=True, align="C")
        pdf.set_font("Helvetica", "", 9)
        pdf.cell(0, 6, "Clinical Diagnostic Report", ln=True, align="C")
        pdf.cell(0, 5, f"Report ID: PEA-{d['patient_id']}-{int(time.time())}", ln=True, align="C")

        pdf.ln(12)
        pdf.set_text_color(30, 41, 59)
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(0, 7, "PATIENT INFORMATION", ln=True)
        pdf.set_draw_color(226, 232, 240)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)

        pdf.set_font("Helvetica", "", 9)
        for label, value in [("Patient ID", str(d["patient_id"])), ("Age", f"{d['patient_age']} Years"),
                              ("Date", d["timestamp"]), ("Modality", d["modality"])]:
            pdf.set_text_color(100, 116, 139)
            pdf.cell(50, 6, f"{label}:")
            pdf.set_text_color(30, 41, 59)
            pdf.cell(0, 6, safe_pdf_text(value), ln=True)

        pdf.ln(5)
        pdf.set_text_color(30, 41, 59)
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(0, 7, "AI ANALYSIS RESULTS", ln=True)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)

        pdf.set_font("Helvetica", "", 9)
        for label, value in [("Model", d["model_used"]), ("Verdict", d["verdict"]),
                              ("Confidence", f"{d['confidence']:.1f}%"), ("Inference", f"{d['inference_time']}s"),
                              ("Localization", d["localization"]), ("Hardware", "RK3588 NPU (INT8)")]:
            pdf.set_text_color(100, 116, 139)
            pdf.cell(50, 6, f"{label}:")
            pdf.set_text_color(30, 41, 59)
            pdf.cell(0, 6, safe_pdf_text(value), ln=True)

        pdf.ln(5)
        pdf.set_text_color(30, 41, 59)
        pdf.set_font("Helvetica", "B", 11)
        pdf.cell(0, 7, "CLINICAL ASSESSMENT", ln=True)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)

        is_tb_report = "Tuberculosis" in d.get("model_used", "") or d.get("modality", "") == "DX"
        pdf.set_font("Helvetica", "", 9)
        if is_tb_report:
            for label, value in [("TB Severity (WHO)", d.get("bi_rads", "N/A")),
                                  ("Lung Zone", d.get("acr_density", "N/A")),
                                  ("TB Status", d.get("verdict", "Unknown"))]:
                pdf.set_text_color(100, 116, 139)
                pdf.cell(50, 6, f"{label}:")
                pdf.set_text_color(30, 41, 59)
                pdf.cell(0, 6, safe_pdf_text(value), ln=True)
            if "Positive" in d.get("verdict", ""):
                pdf.ln(3)
                pdf.set_fill_color(239, 68, 68)
                pdf.set_text_color(255, 255, 255)
                pdf.set_font("Helvetica", "B", 9)
                pdf.cell(0, 8, "URGENT: Refer to district TB centre. GeneXpert required.", ln=True, fill=True, align="C")
                pdf.set_text_color(30, 41, 59)
        else:
            pdf.set_text_color(100, 116, 139)
            pdf.cell(50, 6, "BI-RADS:")
            pdf.set_text_color(30, 41, 59)
            pdf.cell(0, 6, safe_pdf_text(d["bi_rads"]), ln=True)
            pdf.set_text_color(100, 116, 139)
            pdf.cell(50, 6, "ACR Density:")
            pdf.set_text_color(30, 41, 59)
            pdf.cell(0, 6, safe_pdf_text(d["acr_density"]), ln=True)

        pdf.ln(15)
        pdf.set_draw_color(226, 232, 240)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(3)
        pdf.set_font("Helvetica", "", 7)
        pdf.set_text_color(148, 163, 184)
        pdf.cell(0, 5, "Pink Edge AI v5.3 - Alibaba Cloud AI Hackathon 2026", ln=True, align="C")
        pdf.cell(0, 5, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", ln=True, align="C")

        return bytes(pdf.output(dest="S"))
    except Exception as e:
        st.error(f"PDF Generation Error: {e}")
        return None


IMAGE GENERATION

In [16]:
@st.cache_data(max_entries=20)
def generate_mammogram(size=512, seed=42):
    img = np.zeros((size, size), dtype=np.float32)
    xx, yy = np.meshgrid(np.arange(size), np.arange(size))
    cx, cy = size // 2, int(size * 0.58)
    rx, ry = size * 0.4, size * 0.5
    d = np.sqrt(((xx - cx) / rx) ** 2 + ((yy - cy) / ry) ** 2)
    mask = d < 1.0
    rng = np.random.RandomState(seed)
    img = np.where(mask, 90 + (1 - d) * 70, 0) + rng.normal(0, 12, (size, size)) * mask
    mcx, mcy = int(size * 0.68), int(size * 0.35)
    img[((xx - mcx) ** 2 + (yy - mcy) ** 2) < 22 ** 2] += 35
    img8 = np.clip(img, 0, 255).astype(np.uint8)
    for _ in range(8):
        sx, sy = rng.randint(80, size - 80, 2)
        l, a = rng.randint(30, 70), rng.uniform(0, 2 * np.pi)
        cv2.line(img8, (sx, sy), (int(sx + l * np.cos(a)), int(sy + l * np.sin(a))), rng.randint(120, 160), 1)
    return img8


@st.cache_data(max_entries=20)
def generate_xray(size=512, seed=42):
    img = np.zeros((size, size), dtype=np.float32) + 40
    cv2.ellipse(img, (size // 2, size // 2), (int(size * 0.35), int(size * 0.45)), 0, 0, 360, 180, -1)
    rng = np.random.RandomState(seed)
    img += rng.normal(0, 15, (size, size))
    mcx, mcy = int(size * 0.35), int(size * 0.30)
    img[((np.arange(size)[:, None] - mcx) ** 2 + (np.arange(size)[None, :] - mcy) ** 2) < 15 ** 2] += 80
    return np.clip(img, 0, 255).astype(np.uint8)


@st.cache_data(max_entries=20)
def generate_ultrasound(size=512, seed=42):
    rng = np.random.RandomState(seed)
    img = np.zeros((size, size), dtype=np.float32) + 50 + rng.normal(0, 25, (size, size))
    cv2.circle(img, (size // 2, size // 2), int(size * 0.2), 120, -1)
    return np.clip(img, 0, 255).astype(np.uint8)


def draw_bbox(image, model_type, result_data=None):
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    else:
        image = image.copy()
    h, w = image.shape[:2]

    is_critical = False
    confidence_str = "94.2%"
    label = "Analyzing..."

    if result_data:
        is_critical = result_data.get("is_critical", False)
        confidence_str = f"{result_data['confidence']:.1f}%"

    if is_critical:
        color = (236, 72, 153)  # Pink for danger
    else:
        color = (16, 185, 129)  # Green for OK

    if "Mammography" in model_type:
        center = (int(w * 0.68), int(h * 0.35))
        box_size = (int(w * 0.15), int(h * 0.12))
        angle = 35
        box = cv2.boxPoints((center, box_size, angle)).astype(int)
        cv2.drawContours(image, [box], 0, color, 2)
        for pt in box:
            cv2.circle(image, tuple(pt), 5, color, -1)
        if is_critical:
            label = f"YOLOv8-OBB: Malignant ({confidence_str})"
        else:
            label = f"YOLOv8-OBB: Benign ({confidence_str})"
    elif "Tuberculosis" in model_type:
        center = (int(w * 0.35), int(h * 0.30))
        box_size = (int(w * 0.12), int(h * 0.10))
        box = cv2.boxPoints((center, box_size, 0)).astype(int)
        cv2.rectangle(image, (box[0][0], box[0][1]), (box[2][0], box[2][1]), color, 2)
        if is_critical:
            label = f"TB Classifier: Active Lesion ({confidence_str})"
        else:
            label = f"TB Classifier: Clear ({confidence_str})"
    else:
        center = (int(w * 0.50), int(h * 0.50))
        cv2.circle(image, center, int(w * 0.15), color, 2)
        label = f"Fetal Health: Normal ({confidence_str})"

    font = cv2.FONT_HERSHEY_SIMPLEX
    text_size = cv2.getTextSize(label, font, 0.5, 1)[0]
    lx = max(0, center[0] - text_size[0] // 2)
    ly = max(25, center[1] - int(w * 0.15) - 10)
    cv2.rectangle(image, (lx, ly - text_size[1] - 8), (lx + text_size[0] + 12, ly + 5), (0, 0, 0), -1)
    cv2.putText(image, label, (lx + 6, ly), font, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.drawMarker(image, center, (0, 230, 255), cv2.MARKER_CROSS, 20, 1)
    return image


def load_image(model_type, uploaded_file=None, seed=42):
    if uploaded_file is not None:
        file_bytes = np.asarray(bytearray(uploaded_file.read()), dtype=np.uint8)
        img = cv2.imdecode(file_bytes, cv2.IMREAD_GRAYSCALE)
        if img is not None:
            return cv2.resize(img, (512, 512))
    if "Mammography" in model_type:
        return generate_mammogram(512, seed)
    elif "Tuberculosis" in model_type:
        return generate_xray(512, seed)
    else:
        return generate_ultrasound(512, seed)


def ensure_dirs():
    os.makedirs("models", exist_ok=True)
    for f in ["yolov8_obb_int8.pt", "tb_classifier.pt", "fetal_health.pt"]:
        p = f"models/{f}"
        if not os.path.exists(p):
            with open(p, "w") as fh:
                fh.write("Dummy Model")

2026-09-12 11:16:43.327 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-12 11:16:43.344 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-09-12 11:16:43.356 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


CSS

In [19]:
CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=JetBrains+Mono:wght@400;500;600&family=Noto+Nastaliq+Urdu:wght@400;500;600&display=swap');

:root {
    --bg: #0a0e1a;
    --surface: #111827;
    --surface-alt: #1e293b;
    --surface-hover: #334155;
    --border: #1e293b;
    --border-light: #334155;
    --text: #f1f5f9;
    --text-muted: #94a3b8;
    --text-light: #64748b;
    --primary: #14b8a6;
    --primary-light: #2dd4bf;
    --primary-dark: #0d9488;
    --primary-bg: rgba(20, 184, 166, 0.1);
    --accent: #ec4899;
    --accent-light: #f472b6;
    --accent-dark: #db2777;
    --accent-bg: rgba(236, 72, 153, 0.1);
    --success: #10b981;
    --success-bg: rgba(16, 185, 129, 0.1);
    --warning: #f59e0b;
    --warning-bg: rgba(245, 158, 11, 0.1);
    --danger: #ef4444;
    --danger-bg: rgba(239, 68, 68, 0.1);
    --shadow-sm: 0 1px 3px rgba(0,0,0,0.3);
    --shadow: 0 4px 6px rgba(0,0,0,0.3);
    --shadow-md: 0 4px 12px rgba(0,0,0,0.4);
    --shadow-lg: 0 10px 25px rgba(0,0,0,0.5);
    --radius: 14px;
    --radius-sm: 10px;
    --radius-xs: 6px;
}

.stApp { background: var(--bg); color: var(--text); font-family: 'Inter', -apple-system, sans-serif; }
.block-container { padding-top: 4rem !important; padding-bottom: 2rem !important; max-width: 1300px !important; }

section[data-testid="stSidebar"] { background: var(--surface) !important; border-right: 1px solid var(--border) !important; }
section[data-testid="stSidebar"] > div:first-child { padding-top: 20px !important; padding-left: 18px !important; padding-right: 18px !important; }
section[data-testid="stSidebar"] h1, section[data-testid="stSidebar"] h2, section[data-testid="stSidebar"] h3, section[data-testid="stSidebar"] h4 { color: var(--primary-light) !important; }

.sidebar-brand { background: linear-gradient(135deg, var(--primary) 0%, var(--accent) 100%); border-radius: var(--radius-sm); padding: 16px; margin-bottom: 16px; display: flex; align-items: center; gap: 10px; box-shadow: 0 4px 12px rgba(20, 184, 166, 0.2); }
.sidebar-brand .logo { width: 40px; height: 40px; background: rgba(255,255,255,0.2); border-radius: 10px; display: flex; align-items: center; justify-content: center; font-size: 1.3rem; }
.sidebar-brand .text { color: #fff; }
.sidebar-brand .text .name { font-weight: 800; font-size: 1rem; }
.sidebar-brand .text .sub { font-size: 0.7rem; opacity: 0.85; }

h1, h2, h3, h4 { color: var(--text) !important; font-family: 'Inter', sans-serif !important; font-weight: 700 !important; }
h1 { border: none !important; padding: 0 !important; font-size: 1.6rem !important; }
h2 { font-size: 1.2rem !important; margin-top: 1.2rem !important; }
h3 { font-size: 0.78rem !important; color: var(--text-muted) !important; text-transform: uppercase; letter-spacing: 0.8px; font-weight: 700 !important; }

.page-header { background: linear-gradient(135deg, var(--primary) 0%, var(--accent) 100%); border-radius: var(--radius); padding: 28px 32px; margin-bottom: 20px; box-shadow: var(--shadow-md); }
.page-header h1 { color: #fff !important; font-size: 1.5rem !important; font-weight: 800 !important; margin: 0 !important; }
.page-header p { color: rgba(255,255,255,0.85); font-size: 0.85rem; margin: 6px 0 0 0; }

.badge { display: inline-flex; align-items: center; gap: 5px; padding: 5px 12px; border-radius: 20px; font-size: 0.72rem; font-weight: 600; }
.badge-offline { background: var(--success-bg); color: var(--success); border: 1px solid rgba(16, 185, 129, 0.3); }
.badge-cloud { background: var(--warning-bg); color: var(--warning); border: 1px solid rgba(245, 158, 11, 0.3); }

.card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 20px; margin: 8px 0; box-shadow: var(--shadow-sm); }
.card-accent { border-left: 4px solid var(--accent); }
.card-primary { border-left: 4 غربية
.card-success { border-left: 4px solid var(--success); }
.card-danger { border-left: 4px solid var(--danger); }

.metric-tile { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius-sm); padding: 16px; text-align: center; box-shadow: var(--shadow-sm); transition: transform 0.15s, box-shadow 0.15s; }
.metric-tile:hover { transform: translateY(-2px); box-shadow: var(--shadow-md); border-color: var(--primary); }
.metric-tile .label { color: var(--text-muted); font-size: 0.68rem; text-transform: uppercase; letter-spacing: 0.8px; font-weight: 600; }
.metric-tile .value { font-size: 1.6rem; font-weight: 800; margin-top: 6px; }
.metric-tile .value.primary { color: var(--primary-light); }
.metric-tile .value.accent { color: var(--accent-light); }
.metric-tile .value.success { color: var(--success); }
.metric-tile .value.danger { color: var(--danger); }
.metric-tile .value.warning { color: var(--warning); }

.image-viewer { background: #000; border: 2px solid var(--border); border-radius: var(--radius); overflow: hidden; position: relative; }
.image-viewer .img-overlay { position: absolute; top: 10px; left: 10px; background: rgba(0,0,0,0.7); backdrop-filter: blur(8px); border-radius: var(--radius-xs); padding: 5px 10px; font-size: 0.72rem; color: #fff; font-family: 'JetBrains Mono', monospace; }

.dicom-grid { background: var(--surface-alt); border: 1px solid var(--border); border-radius: var(--radius-sm); padding: 16px; display: grid; grid-template-columns: 1fr 1fr; gap: 12px; }
.dicom-item .label { color: var(--text-light); font-size: 0.68rem; text-transform: uppercase; letter-spacing: 0.5px; font-weight: 600; }
.dicom-item .value { color: var(--text); font-size: 0.88rem; font-weight: 600; margin-top: 3px; }

.verdict-box { border-radius: var(--radius-sm); padding: 24px; text-align: center; margin: 8px 0; }
.verdict-box.danger { background: linear-gradient(135deg, var(--danger-bg) 0%, rgba(239,68,68,0.03) 100%); border: 2px solid rgba(239, 68, 68, 0.3); }
.verdict-box.success { background: linear-gradient(135deg, var(--success-bg) 0%, rgba(16,185,129,0.03) 100%); border: 2px solid rgba(16, 185, 129, 0.3); }
.verdict-box .v-icon { font-size: 1.8rem; margin-bottom: 6px; }
.verdict-box .v-title { font-size: 1.5rem; font-weight: 800; }
.verdict-box.danger .v-title { color: var(--danger); }
.verdict-box.success .v-title { color: var(--success); }
.verdict-box .v-sub { font-size: 0.88rem; margin-top: 4px; color: var(--text-muted); }

.console-log { background: #060a13; border: 1px solid var(--border-light); border-radius: var(--radius-sm); padding: 16px; font-family: 'JetBrains Mono', monospace; font-size: 0.76rem; line-height: 1.8; max-height: 300px; overflow-y: auto; white-space: pre-wrap; word-wrap: break-word; }
.console-log .c-dicom { color: #60a5fa; }
.console-log .c-npu { color: #34d399; }
.console-log .c-gsm { color: #22d3ee; }
.console-log .c-cloud { color: #fbbf24; }
.console-log .c-cache { color: #f472b6; }
.console-log .c-muted { color: #475569; }
.console-log::-webkit-scrollbar { width: 5px; }
.console-log::-webkit-scrollbar-track { background: transparent; }
.console-log::-webkit-scrollbar-thumb { background: var(--border-light); border-radius: 3px; }

.hw-panel { background: var(--surface-alt); border: 1px solid var(--border); border-radius: var(--radius-sm); padding: 14px; font-family: 'JetBrains Mono', monospace; font-size: 0.76rem; }
.hw-row { display: flex; justify-content: space-between; align-items: center; padding: 5px 0; }
.hw-row .hw-label { color: var(--text-muted); }
.hw-row .hw-value { font-weight: 600; }

.alert-card { border-radius: var(--radius-sm); padding: 16px; margin: 6px 0; border-left: 4px solid; box-shadow: var(--shadow-sm); }
.alert-card.critical { background: var(--danger-bg); border-color: var(--danger); }
.alert-card.ok { background: var(--success-bg); border-color: var(--success); }

.data-table { width: 100%; border-collapse: collapse; font-size: 0.8rem; font-family: 'JetBrains Mono', monospace; border-radius: var(--radius-sm); overflow: hidden; }
.data-table th { background: var(--surface-alt); color: var(--primary-light); padding: 12px 14px; text-align: left; border-bottom: 2px solid var(--border-light); font-weight: 700; font-size: 0.7rem; text-transform: uppercase; letter-spacing: 0.5px; }
.data-table td { padding: 10px 14px; border-bottom: 1px solid var(--border); color: var(--text); }
.data-table tr:hover { background: var(--surface-alt); }

.cloud-card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 20px; margin: 8px 0; box-shadow: var(--shadow-sm); border-top: 4px solid var(--primary); }
.cloud-card .c-header { display: flex; align-items: center; gap: 10px; margin-bottom: 12px; }
.cloud-card .c-icon { font-size: 1.2rem; }
.cloud-card .c-title { font-weight: 700; color: var(--primary-light); font-size: 0.92rem; }
.cloud-card .c-desc { font-size: 0.8rem; color: var(--text-muted); margin-bottom: 12px; line-height: 1.5; }
.cloud-card .c-row { display: flex; justify-content: space-between; padding: 5px 0; font-size: 0.78rem; font-family: 'JetBrains Mono', monospace; }
.cloud-card .c-label { color: var(--text-muted); }
.cloud-card .c-value { font-weight: 600; }

.arch-box { background: #060a13; border: 1px solid var(--border-light); border-radius: var(--radius-sm); padding: 18px; font-family: 'JetBrains Mono', monospace; font-size: 0.72rem; line-height: 1.8; color: #94a3b8; }

.section-header { display: flex; align-items: center; gap: 10px; margin: 16px 0 10px; }
.section-header .s-icon { width: 30px; height: 30px; border-radius: 8px; display: flex; align-items: center; justify-content: center; font-size: 0.95rem; background: var(--primary-bg); border: 1px solid rgba(20, 184, 166, 0.2); }
.section-header .s-title { font-size: 1.05rem; font-weight: 700; color: var(--text); }

.stButton > button { background: var(--surface-alt) !important; color: var(--text) !important; border: 1px solid var(--border-light) !important; border-radius: var(--radius-sm) !important; font-family: 'Inter', sans-serif !important; font-weight: 600 !important; font-size: 0.82rem !important; padding: 9px 16px !important; transition: all 0.2s !important; }
.stButton > button:hover { background: var(--surface-hover) !important; border-color: var(--primary) !important; color: var(--primary-light) !important; box-shadow: 0 0 16px rgba(20, 184, 166, 0.2) !important; }

.stDownloadButton > button { background: var(--accent) !important; color: #fff !important; border: none !important; border-radius: var(--radius-sm) !important; font-weight: 600 !important; font-size: 0.82rem !important; box-shadow: 0 1px 3px rgba(236, 72, 153, 0.25) !important; }
.stDownloadButton > button:hover { background: var(--accent-dark) !important; box-shadow: 0 4px 12px rgba(236, 72, 153, 0.35) !important; transform: translateY(-1px) !important; }

.stSelectbox > div > div { background: var(--surface-alt) !important; border: 1px solid var(--border-light) !important; border-radius: var(--radius-sm) !important; color: var(--text) !important; }
.stSelectbox > div > div:hover { border-color: var(--primary) !important; }

.stFileUploader > div { background: var(--surface-alt) !important; border: 1px dashed var(--border-light) !important; border-radius: var(--radius-sm) !important; }
.stFileUploader > div:hover { border-color: var(--accent) !important; }

.stCheckbox > label { color: var(--text) !important; }
.stCheckbox > label > div:first-child { border-color: var(--border-light) !important; background: var(--surface-alt) !important; }
.stCheckbox > label > div:first-child[data-checked="true"] { background: var(--primary) !important; border-color: var(--primary) !important; }

.stRadio > label { color: var(--text) !important; }
.stRadio > label > div:first-child { border-color: var(--border-light) !important; }
.stRadio > label > div:first-child[data-checked="true"] { background: var(--primary) !important; border-color: var(--primary) !important; }

.stMetric { background: var(--surface) !important; border: 1px solid var(--border) !important; border-radius: var(--radius-sm) !important; padding: 12px !important; box-shadow: var(--shadow-sm) !important; }
.stMetric label { color: var(--text-muted) !important; font-size: 0.75rem !important; }
.stMetric [data-testid="stMetricValue"] { color: var(--text) !important; font-weight: 700 !important; }

.stAlert { border: none !important; border-radius: var(--radius-sm) !important; }
.stAlert div[data-baseweb="notification"] { background: var(--surface) !important; border: 1px solid var(--border) !important; border-radius: var(--radius-sm) !important; }

.stTabs [data-baseweb="tab-list"] { gap: 8px; background: transparent !important; padding: 0 !important; border: none !important; border-bottom: 1px solid var(--border) !important; border-radius: 0 !important; }
.stTabs [data-baseweb="tab"] { background: transparent !important; border: none !important; border-radius: 0 !important; padding: 10px 20px !important; font-weight: 600 !important; color: var(--text-muted) !important; transition: color 0.2s !important; }
.stTabs [data-baseweb="tab"]:hover { background: transparent !important; color: var(--primary-light) !important; }
.stTabs [aria-selected="true"] { background: transparent !important; color: var(--primary) !important; }
.stTabs [data-baseweb="tab-highlight"] { background-color: transparent !important; height: 0px !important; }
.stTabs [data-baseweb="tab-border"] { border: none !important; border-bottom: 2px solid var(--primary) !important; }

.app-footer { text-align: center; padding: 24px; color: var(--text-light); font-size: 0.72rem; border-top: 1px solid var(--border); margin-top: 24px; }
.urdu { direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu', serif; }

@media (max-width: 768px) { .dicom-grid { grid-template-columns: 1fr; } .block-container { padding-left: 12px; padding-right: 12px; } }
</style>
"""

SESSION STATE

In [20]:

def init_state():
    defaults = {
        "log_entries": [], "inference_done": False, "npu_active": False,
        "last_uploaded": None, "last_model": None, "sms_alerts": [],
        "pat_id": random.randint(10000000, 99999999),
        "pat_age": random.randint(30, 70),
        "hw_stats": {"load": "34%", "power": "6.2W", "temp": "42C"},
        "net_stats": {"signal": "-85 dBm", "bhus": 14, "module": "ONLINE"},
        "urdu_mode": False, "show_bbox": True,
        "network_mode": "Fully Offline",
        "bi_rads_selected": None, "acr_density_selected": None,
        "cache_saved": False, "sync_done": False,
        "iot_messages": [], "oss_uploads": [], "acr_status": None,
        "current_result": None,
        "inference_latency": 9.4,
    }
    for k, v in defaults.items():
        if k not in st.session_state:
            st.session_state[k] = v

SIDEBAR

In [21]:
def render_sidebar():
    with st.sidebar:
        st.markdown("""
        <div class="sidebar-brand">
            <div class="logo">🩸</div>
            <div class="text">
                <div class="name">Pink Edge AI</div>
                <div class="sub">Clinical Intelligence Platform</div>
            </div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"#### {t('Language')}")
        lc1, lc2 = st.columns(2)
        with lc1:
            if st.button("🇬🇧 EN", use_container_width=True, key="lang_en",
                         type="primary" if not st.session_state.urdu_mode else "secondary"):
                st.session_state.urdu_mode = False
                st.rerun()
        with lc2:
            if st.button("🇵🇰 اردو", use_container_width=True, key="lang_ur",
                         type="primary" if st.session_state.urdu_mode else "secondary"):
                st.session_state.urdu_mode = True
                st.rerun()

        st.markdown("---")

        st.markdown(f"#### {t('Network Mode')}")
        network_mode = st.radio(
            t("Network Mode"),
            ["Fully Offline", "GSM Failover"],
            label_visibility="collapsed",
            key="net_radio",
            index=0 if st.session_state.network_mode == "Fully Offline" else 1
        )
        st.session_state.network_mode = network_mode

        if "GSM" in network_mode:
            st.markdown('<span class="badge badge-cloud">☁️ Alibaba Cloud IoT Active</span>', unsafe_allow_html=True)
        else:
            st.markdown('<span class="badge badge-offline">🔒 100% Offline</span>', unsafe_allow_html=True)

        st.markdown("---")

        st.markdown(f"#### {t('Select AI Model')}")
        selected_model = st.selectbox(t("Select AI Model"), MODELS, label_visibility="collapsed", key="model_sel")

        st.markdown(f"#### {t('Upload Patient Scan')}")
        uploaded_file = st.file_uploader(t("Upload Patient Scan"), type=["jpg", "jpeg", "png"], label_visibility="collapsed", key="file_up")

        # Reset state when model changes
        if st.session_state.last_model != selected_model:
            st.session_state.inference_done = False
            st.session_state.log_entries = []
            st.session_state.last_model = selected_model
            st.session_state.cache_saved = False
            st.session_state.current_result = None

        st.markdown("---")

        st.markdown(f"#### {t('Actions')}")

        if st.button(f"📡 {t('Ingest DICOM')}", use_container_width=True, key="btn_dicom"):
            ts = time.strftime("%H:%M:%S")
            st.session_state.log_entries.append(f"[{ts}] [DICOM] Ingesting raw scan via LAN... Success.")
            st.success(t("DICOM scan ingested successfully!"))

        if st.button(f"▶️ {t('Run Triage')}", use_container_width=True, key="btn_triage"):
            st.session_state.npu_active = True
            with st.spinner("Running INT8 on NPU..."):
                time.sleep(2)

            # FIX: Generate FRESH patient data each time Run Triage is clicked
            new_patient = generate_new_patient()
            st.session_state.pat_id = new_patient["pat_id"]
            st.session_state.pat_age = new_patient["pat_age"]
            st.session_state.hw_stats = generate_hw_stats()
            st.session_state.net_stats = generate_net_stats()

            # Generate varied result based on model type
            if "Mammography" in selected_model:
                result = generate_mammography_result()
            elif "Tuberculosis" in selected_model:
                # Load image here for inference (sidebar has no current_image variable)
                _tb_img_seed = int(time.time()) % 10000
                _tb_image = load_image(selected_model, uploaded_file, _tb_img_seed)
                result = generate_tb_result(image_np=_tb_image)
            else:
                result = generate_fetal_result()

            st.session_state.current_result = result
            st.session_state.bi_rads_selected = result["bi_rads"]
            st.session_state.acr_density_selected = result["acr"]
            st.session_state.inference_latency = round(random.uniform(7.8, 12.2), 1)

            ts = time.strftime("%H:%M:%S")
            pat_id = st.session_state.pat_id
            sms_payload = f"ID:{pat_id}|LOC:29.344|{result['sms']}"

            st.session_state.log_entries = [
                f"[{ts}] [DICOM] Ingesting raw scan via LAN... Success.",
                f"[{ts}] [NPU] INT8 Quantized Core for {selected_model.split('(')[0].strip()}... Complete ({st.session_state.inference_latency}s).",
                f"[{ts}] [GSM] Compressing to 140-char string...",
                f'[{ts}] [GSM] Broadcasted: "{sms_payload}" -> Allied Hospital Hub.',
            ]
            if "GSM" in st.session_state.network_mode:
                iot_id = f"IOT-{random.randint(100000, 999999)}"
                st.session_state.log_entries.append(f"[{ts}] [Alibaba IoT] Queued - ID: {iot_id} -> Table Store")
                st.session_state.iot_messages.append({"time": ts, "id": pat_id, "payload": sms_payload, "iot_id": iot_id})

            st.session_state.sms_alerts.insert(0, {
                "time": ts, "id": pat_id, "type": selected_model.split("(")[0].strip(),
                "payload": sms_payload, "status": "Pending Review",
                "is_critical": result["is_critical"]
            })
            st.session_state.inference_done = True
            st.session_state.npu_active = False
            st.session_state.cache_saved = False
            st.session_state.hw_stats["load"] = "97%"

        if st.session_state.inference_done:
            if st.button(f"💾 {t('Save to Cache')}", use_container_width=True, key="btn_cache"):
                mod_map = {"Mammography (YOLOv8-OBB)": "MG", "Tuberculosis (Chest X-Ray)": "DX", "Maternal Health (Ultrasound)": "US"}
                result = st.session_state.current_result

                cache_data = {
                    "patient_id": st.session_state.pat_id, "patient_age": st.session_state.pat_age,
                    "modality": mod_map.get(selected_model, "Unknown"),
                    "model_used": selected_model.split("(")[0].strip(),
                    "bi_rads": st.session_state.bi_rads_selected or "N/A",
                    "acr_density": st.session_state.acr_density_selected or "N/A",
                    "verdict": result["verdict"] if result else "Unknown",
                    "localization": result["loc"] if result else "Unknown",
                    "confidence": result["confidence"] if result else 0.0,
                    "inference_time": st.session_state.inference_latency,
                    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "network_mode": st.session_state.network_mode, "synced": 0,
                }
                save_to_cache(cache_data)
                st.session_state.cache_saved = True
                ts = time.strftime("%H:%M:%S")
                st.session_state.log_entries.append(f"[{ts}] [SQLite] Report cached (ID: {st.session_state.pat_id}).")
                st.success(t("Report safely cached to offline hardware node."))

        if "GSM" in st.session_state.network_mode and get_unsynced_count() > 0:
            if st.button(f"☁️ {t('Sync to Cloud')}", use_container_width=True, key="btn_sync"):
                unsynced = get_unsynced_reports()
                with st.spinner("Syncing to Alibaba Cloud..."):
                    iot_results = simulate_iot_sync(unsynced)
                    oss_results = [{"report_id": r[0], **simulate_oss_upload(r)} for r in unsynced]
                    mark_as_synced([r[0] for r in unsynced])
                st.session_state.iot_messages = iot_results
                st.session_state.oss_uploads = oss_results
                st.session_state.sync_done = True
                ts = time.strftime("%H:%M:%S")
                st.session_state.log_entries.append(f"[{ts}] [Alibaba IoT] {len(iot_results)} messages synced to Table Store.")
                high = sum(1 for r in oss_results if r.get("high"))
                if high > 0:
                    st.session_state.log_entries.append(f"[{ts}] [Alibaba OSS] {high} high-risk patches uploaded.")
                st.success(t("Sync complete! Reports uploaded to Alibaba Cloud."))

        if "GSM" in st.session_state.network_mode:
            if st.button(f"🔍 {t('Check OTA')}", use_container_width=True, key="btn_ota"):
                with st.spinner("Checking ACR..."):
                    time.sleep(1.5)
                    st.session_state.acr_status = simulate_acr_check()
                if st.session_state.acr_status:
                    st.info(f"📦 {st.session_state.acr_status['current']} -> {st.session_state.acr_status['available']}")

        st.markdown("---")

        st.session_state.show_bbox = st.checkbox(t("Show Detection Overlay"), value=st.session_state.show_bbox, key="bbox_toggle")

        st.markdown("---")

        st.markdown(f"#### {t('Hardware Diagnostics')}")
        load = st.session_state.hw_stats["load"]
        load_color = "#ef4444" if "97" in load else "#10b981"

        st.markdown(f"""
        <div class="hw-panel">
            <div class="hw-row"><span class="hw-label">🔌 RK3588</span><span class="hw-value" style="color:#10b981;">● ACTIVE</span></div>
            <div class="hw-row"><span class="hw-label">⚡ NPU Load</span><span class="hw-value" style="color:{load_color};">{load}</span></div>
            <div class="hw-row"><span class="hw-label">🔋 Power</span><span class="hw-value" style="color:#2dd4bf;">{st.session_state.hw_stats['power']}</span></div>
            <div class="hw-row"><span class="hw-label">🌡️ Temp</span><span class="hw-value" style="color:#2dd4bf;">{st.session_state.hw_stats['temp']}</span></div>
            <div class="hw-row"><span class="hw-label">📦 Model</span><span class="hw-value" style="font-size:0.7rem;">{selected_model.split('(')[0].strip()}</span></div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("---")
        if st.button(f"🔄 {t('Reset Session')}", use_container_width=True, key="btn_reset"):
            st.session_state.clear()
            st.rerun()

    return selected_model, uploaded_file


VIEW: DASHBOARD

In [22]:
def render_dashboard(selected_model, uploaded_file):
    st.markdown("""
    <div class="page-header">
        <h1>🩸 Pink Edge AI</h1>
        <p>Clinical Intelligence Platform • Edge AI on Rockchip RK3588 NPU • YOLOv8-OBB INT8</p>
    </div>
    """, unsafe_allow_html=True)

    if "GSM" in st.session_state.network_mode:
        st.markdown('<span class="badge badge-cloud">☁️ GSM Failover — Alibaba Cloud Sync Active</span>', unsafe_allow_html=True)
    else:
        st.markdown('<span class="badge badge-offline">🔒 100% Offline — Zero Cloud Dependency</span>', unsafe_allow_html=True)

    st.markdown("")

    # Use time-based seed for image variety when no upload
    img_seed = int(time.time()) % 10000 if not uploaded_file else hash(uploaded_file.name) % 10000
    current_image = load_image(selected_model, uploaded_file, img_seed)
    display_name = uploaded_file.name if uploaded_file else "Generated Placeholder"

    col_img, col_meta = st.columns([3, 2])

    with col_img:
        st.markdown(f"""
        <div class="section-header">
            <div class="s-icon">🔬</div>
            <div class="s-title">{t('Medical Image Analysis')}</div>
        </div>
        """, unsafe_allow_html=True)

        if st.session_state.inference_done and st.session_state.current_result:
            result = st.session_state.current_result
            if st.session_state.show_bbox:
                annotated = draw_bbox(current_image, selected_model, result)
                st.markdown('<div class="image-viewer">', unsafe_allow_html=True)
                st.image(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB), use_container_width=True)
                st.markdown(f'<div class="img-overlay">{display_name} | Detection: ACTIVE</div>', unsafe_allow_html=True)
                st.markdown('</div>', unsafe_allow_html=True)
            else:
                st.markdown('<div class="image-viewer">', unsafe_allow_html=True)
                st.image(current_image, use_container_width=True)
                st.markdown(f'<div class="img-overlay">{display_name} | Overlay: OFF</div>', unsafe_allow_html=True)
                st.markdown('</div>', unsafe_allow_html=True)

            mc1, mc2, mc3 = st.columns(3)
            with mc1:
                st.markdown('<div class="metric-tile"><div class="label">Model</div><div class="value primary" style="font-size:1rem;">YOLOv8</div></div>', unsafe_allow_html=True)
            with mc2:
                conf_str = f"{result['confidence']:.1f}%"
                val_class = "accent" if result["is_critical"] else "success"
                st.markdown(f'<div class="metric-tile"><div class="label">{t("Confidence")}</div><div class="value {val_class}">{conf_str}</div></div>', unsafe_allow_html=True)
            with mc3:
                latency_str = f"{st.session_state.inference_latency}s"
                st.markdown(f'<div class="metric-tile"><div class="label">{t("Latency")}</div><div class="value primary">{latency_str}</div></div>', unsafe_allow_html=True)
        else:
            st.markdown('<div class="image-viewer">', unsafe_allow_html=True)
            st.image(current_image, use_container_width=True)
            st.markdown(f'<div class="img-overlay">{display_name} | Awaiting Analysis</div>', unsafe_allow_html=True)
            st.markdown('</div>', unsafe_allow_html=True)
            st.info(f"📋 {t('Awaiting Analysis')} — {t('Run triage to begin')}")

    with col_meta:
        st.markdown(f"""
        <div class="section-header">
            <div class="s-icon">📋</div>
            <div class="s-title">{t('DICOM Metadata')}</div>
        </div>
        """, unsafe_allow_html=True)

        mod_map = {"Mammography (YOLOv8-OBB)": "MG (Mammography)", "Tuberculosis (Chest X-Ray)": "DX (Digital Radiography)", "Maternal Health (Ultrasound)": "US (Ultrasound)"}
        bp_map = {"Mammography (YOLOv8-OBB)": "BREAST", "Tuberculosis (Chest X-Ray)": "CHEST", "Maternal Health (Ultrasound)": "ABDOMEN"}

        st.markdown(f"""
        <div class="dicom-grid">
            <div class="dicom-item"><div class="label">{t('Patient ID')}</div><div class="value">{st.session_state.pat_id}</div></div>
            <div class="dicom-item"><div class="label">Age</div><div class="value">{st.session_state.pat_age} Y</div></div>
            <div class="dicom-item"><div class="label">{t('Modality')}</div><div class="value">{mod_map[selected_model]}</div></div>
            <div class="dicom-item"><div class="label">Date</div><div class="value">{datetime.now().strftime('%Y-%m-%d')}</div></div>
            <div class="dicom-item"><div class="label">Body Part</div><div class="value">{bp_map[selected_model]}</div></div>
            <div class="dicom-item"><div class="label">Institution</div><div class="value">Rural BHU FSD</div></div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("")

        if st.session_state.inference_done and st.session_state.current_result:
            result = st.session_state.current_result

            st.markdown(f"""
            <div class="verdict-box {result['css']}">
                <div class="v-icon">{result['vicon']}</div>
                <div class="v-title">{result['verdict']}</div>
                <div class="v-sub">{result['sub']}</div>
            </div>
            """, unsafe_allow_html=True)

            st.markdown(f"""
            <div class="card card-primary" style="padding:16px;">
                <div style="display:flex;justify-content:space-between;padding:5px 0;">
                    <span style="color:var(--text-muted);font-size:0.78rem;">{t('Localization')}</span>
                    <span style="font-weight:600;font-size:0.82rem;">{result['loc']}</span>
                </div>
                <div style="display:flex;justify-content:space-between;padding:5px 0;">
                    <span style="color:var(--text-muted);font-size:0.78rem;">{t('Classification')}</span>
                    <span style="font-weight:600;font-size:0.82rem;">{result['extra']}</span>
                </div>
                <div style="display:flex;justify-content:space-between;padding:5px 0;">
                    <span style="color:var(--text-muted);font-size:0.78rem;">{t('Inference Time')}</span>
                    <span style="font-weight:600;font-size:0.82rem;color:var(--primary-light);">{st.session_state.inference_latency}s</span>
                </div>
                <div style="display:flex;justify-content:space-between;padding:5px 0;">
                    <span style="color:var(--text-muted);font-size:0.78rem;">GSM Alert</span>
                    <span style="font-weight:600;font-size:0.82rem;color:var(--success);">✅ Broadcasted</span>
                </div>
            </div>
            """, unsafe_allow_html=True)

            st.markdown(f"""
            <div class="section-header">
                <div class="s-icon">✏️</div>
                <div class="s-title">{t('Confirm Assessment')}</div>
            </div>
            """, unsafe_allow_html=True)

            if "Tuberculosis" in selected_model:
                # TB uses WHO Severity Index — BI-RADS is a mammography standard, wrong here
                tb_sev_default = result.get("tb_severity", TB_SEVERITY_LEVELS[0])
                tb_sev_idx = TB_SEVERITY_LEVELS.index(tb_sev_default) if tb_sev_default in TB_SEVERITY_LEVELS else 0
                tb_sev_sel = st.selectbox("TB Severity (WHO Index)", TB_SEVERITY_LEVELS,
                                          index=tb_sev_idx, key="tb_sev_dd")
                tb_zone_default = result.get("loc", TB_LUNG_ZONES[0])
                zone_match = next((z for z in TB_LUNG_ZONES if z.lower() in tb_zone_default.lower()),
                                  TB_LUNG_ZONES[0])
                tb_zone_idx = TB_LUNG_ZONES.index(zone_match)
                tb_zone_sel = st.selectbox("Affected Lung Zone", TB_LUNG_ZONES,
                                           index=tb_zone_idx, key="tb_zone_dd")
                # Write back into session state using the schema fields cache expects
                st.session_state.bi_rads_selected   = tb_sev_sel
                st.session_state.acr_density_selected = f"Zone: {tb_zone_sel}"
                if result.get("real_inference"):
                    st.info("🤖 Result from trained tb_classifier.pt model")
                else:
                    st.caption("⚠️ Simulation mode — replace models/tb_classifier.pt with a real trained model")
            else:
                default_br = st.session_state.bi_rads_selected or BI_RADS_OPTIONS[7]
                br_idx = BI_RADS_OPTIONS.index(default_br) if default_br in BI_RADS_OPTIONS else 7
                st.session_state.bi_rads_selected = st.selectbox(t("BI-RADS Assessment"), BI_RADS_OPTIONS, index=br_idx, key="br_dd")

                default_acr = st.session_state.acr_density_selected or ACR_DENSITY_OPTIONS[2]
                acr_idx = ACR_DENSITY_OPTIONS.index(default_acr) if default_acr in ACR_DENSITY_OPTIONS else 2
                st.session_state.acr_density_selected = st.selectbox(t("ACR Breast Density"), ACR_DENSITY_OPTIONS, index=acr_idx, key="acr_dd")

            if result["is_critical"]:
                st.warning(f"⚠️ {t('Immediate Action Required')}: {t('Patient should be referred for specialist consultation')}")
            else:
                st.success(f"✅ {t('Low risk - routine screening')}")

            st.markdown("---")

            report_data = {
                "patient_id": st.session_state.pat_id, "patient_age": st.session_state.pat_age,
                "modality": mod_map.get(selected_model, "Unknown"),
                "model_used": selected_model.split("(")[0].strip(),
                "bi_rads": st.session_state.bi_rads_selected, "acr_density": st.session_state.acr_density_selected,
                "verdict": result["verdict"],
                "localization": result["loc"],
                "confidence": result["confidence"],
                "inference_time": st.session_state.inference_latency,
                "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "network_mode": st.session_state.network_mode,
                "synced": 0 if not st.session_state.cache_saved else 1,
            }

            text_rpt = generate_text_report(report_data)

            dc1, dc2 = st.columns(2)
            with dc1:
                st.download_button(f"📝 {t('Download Text Report')}", text_rpt, file_name=f"report_{st.session_state.pat_id}.txt", mime="text/plain", use_container_width=True, key="dl_txt")
            with dc2:
                if PDF_AVAILABLE:
                    pdf_bytes = generate_pdf_report(report_data)
                    if pdf_bytes:
                        st.download_button(f"📄 {t('Download PDF Report')}", pdf_bytes, file_name=f"report_{st.session_state.pat_id}.pdf", mime="application/pdf", use_container_width=True, key="dl_pdf")
                    else:
                        st.caption("PDF failed")
                else:
                    st.caption("pip install fpdf2 for PDF")

            if st.session_state.cache_saved:
                st.success(t("Report safely cached to offline hardware node."))

    st.markdown("---")
    st.markdown(f"""
    <div class="section-header">
        <div class="s-icon">📊</div>
        <div class="s-title">{t('Telemetry Log')}</div>
    </div>
    """, unsafe_allow_html=True)

    if st.session_state.log_entries:
        log_html = '<div class="console-log">'
        for entry in st.session_state.log_entries:
            if "[DICOM" in entry: cls = "c-dicom"
            elif "[NPU" in entry: cls = "c-npu"
            elif "[GSM" in entry: cls = "c-gsm"
            elif "[Alibaba" in entry: cls = "c-cloud"
            elif "[SQLite" in entry: cls = "c-cache"
            else: cls = "c-muted"
            log_html += f'<div class="{cls}">{entry}</div>'
        log_html += '</div>'
        st.markdown(log_html, unsafe_allow_html=True)
    else:
        st.markdown('<div class="console-log"><span class="c-muted">> System initialized. Awaiting commands...</span></div>', unsafe_allow_html=True)



VIEW: HOSPITAL HUB

In [23]:
def render_hospital_hub():
    st.markdown("""
    <div class="page-header" style="background:linear-gradient(135deg, #14b8a6 0%, #10b981 100%);">
        <h1>🏥 Allied Hospital Faisalabad</h1>
        <p>Urban Receiving Terminal • 2G GSM Critical Alert Monitor</p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("")

    c1, c2 = st.columns([2, 1])

    with c2:
        st.markdown(f"""
        <div class="section-header">
            <div class="s-icon">📡</div>
            <div class="s-title">{t('Network Status')}</div>
        </div>
        """, unsafe_allow_html=True)

        net = st.session_state.net_stats
        nc = "#10b981" if net["module"] == "ONLINE" else "#ef4444"

        st.markdown(f"""
        <div class="hw-panel">
            <div class="hw-row"><span class="hw-label">GSM Module</span><span class="hw-value" style="color:{nc};">● {net['module']}</span></div>
            <div class="hw-row"><span class="hw-label">Signal</span><span class="hw-value" style="color:#10b981;">{net['signal']}</span></div>
            <div class="hw-row"><span class="hw-label">Connected BHUs</span><span class="hw-value" style="color:#2dd4bf;">{net['bhus']} Nodes</span></div>
            <div class="hw-row"><span class="hw-label">Pending Alerts</span><span class="hw-value" style="color:#ef4444;">{len(st.session_state.sms_alerts)}</span></div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown("")
        st.markdown(f"""
        <div class="section-header">
            <div class="s-icon">📈</div>
            <div class="s-title">{t('Stats')}</div>
        </div>
        """, unsafe_allow_html=True)

        mc1, mc2 = st.columns(2)
        with mc1:
            st.markdown(f'<div class="metric-tile"><div class="label">{t("Total Alerts")}</div><div class="value primary">{len(st.session_state.sms_alerts)}</div></div>', unsafe_allow_html=True)
        with mc2:
            critical = sum(1 for a in st.session_state.sms_alerts if a.get("is_critical"))
            st.markdown(f'<div class="metric-tile"><div class="label">{t("Critical Cases")}</div><div class="value danger">{critical}</div></div>', unsafe_allow_html=True)

    with c1:
        st.markdown(f"""
        <div class="section-header">
            <div class="s-icon">🚨</div>
            <div class="s-title">{t('Alert Stream')}</div>
        </div>
        """, unsafe_allow_html=True)

        if not st.session_state.sms_alerts:
            st.markdown("""
            <div class="card card-success" style="text-align:center;padding:50px;">
                <div style="font-size:1.2rem;font-weight:700;color:var(--success);">✅ All Clear</div>
                <div style="color:var(--text-muted);font-size:0.85rem;margin-top:8px;">No critical alerts pending. Waiting for 2G GSM broadcasts from rural edge nodes...</div>
            </div>
            """, unsafe_allow_html=True)
        else:
            for alert in st.session_state.sms_alerts:
                is_crit = alert.get("is_critical", False)
                css = "critical" if is_crit else "ok"
                sc = "#ef4444" if is_crit else "#10b981"

                st.markdown(f"""
                <div class="alert-card {css}">
                    <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px;">
                        <span style="font-weight:800;font-size:1.05rem;color:{sc};">Alert #{alert['id']}</span>
                        <span style="font-size:0.72rem;color:var(--text-light);">{alert['time']}</span>
                    </div>
                    <div style="font-family:'JetBrains Mono',monospace;font-size:0.78rem;color:var(--text);margin-bottom:8px;">
                        <strong>Payload:</strong> "{alert['payload']}"<br>
                        <strong>Type:</strong> {alert['type']}<br>
                        <strong>Status:</strong> <span style="color:{sc};font-weight:600;">{alert['status']}</span>
                    </div>
                    <div style="font-size:0.72rem;color:var(--text-light);">
                        📍 Rural BHU Faisalabad | 🛰️ SIM800L 2G
                    </div>
                </div>
                """, unsafe_allow_html=True)

                if st.button(f"Acknowledge #{alert['id']}", key=f"ack_{alert['id']}", use_container_width=True):
                    alert["status"] = "Acknowledged"
                    st.rerun()

VIEW: CLOUD SYNC

In [24]:

def render_cloud_sync():
    st.markdown("""
    <div class="page-header" style="background:linear-gradient(135deg, #14b8a6 0%, #f59e0b 100%);">
        <h1>☁️ Cloud Sync & Cache</h1>
        <p>Alibaba Cloud Integration • Hybrid-Edge Architecture</p>
    </div>
    """, unsafe_allow_html=True)

    if "GSM" in st.session_state.network_mode:
        st.markdown('<span class="badge badge-cloud">☁️ GSM Failover — Cloud Sync Active</span>', unsafe_allow_html=True)
    else:
        st.markdown('<span class="badge badge-offline">🔒 Fully Offline — Cloud Sync Disabled</span>', unsafe_allow_html=True)

    st.markdown("")

    total = get_cache_count()
    unsynced = get_unsynced_count()
    synced = total - unsynced
    iot_count = len(st.session_state.iot_messages)

    m1, m2, m3, m4 = st.columns(4)
    with m1:
        st.markdown(f'<div class="metric-tile"><div class="label">{t("Cached Reports")}</div><div class="value primary">{total}</div></div>', unsafe_allow_html=True)
    with m2:
        st.markdown(f'<div class="metric-tile"><div class="label">{t("Unsynced")}</div><div class="value danger">{unsynced}</div></div>', unsafe_allow_html=True)
    with m3:
        st.markdown(f'<div class="metric-tile"><div class="label">{t("Synced")}</div><div class="value success">{synced}</div></div>', unsafe_allow_html=True)
    with m4:
        st.markdown(f'<div class="metric-tile"><div class="label">{t("IoT Queue")}</div><div class="value warning">{iot_count}</div></div>', unsafe_allow_html=True)

    st.markdown("")

    cc1, cc2 = st.columns(2)

    with cc1:
        st.markdown(f"""
        <div class="cloud-card">
            <div class="c-header">
                <span class="c-icon">📡</span>
                <span class="c-title">Alibaba Cloud IoT Platform</span>
            </div>
            <div class="c-desc">Link SDK handles 140-char telemetry strings. Reliable offline message queuing for Punjab's unstable 2G networks.</div>
            <div class="c-row"><span class="c-label">Status</span><span class="c-value" style="color:{"#10b981" if "GSM" in st.session_state.network_mode else "#64748b"};">{"● ACTIVE" if "GSM" in st.session_state.network_mode else "○ STANDBY"}</span></div>
            <div class="c-row"><span class="c-label">Endpoint</span><span class="c-value">iot.ap-south-1.aliyuncs.com</span></div>
            <div class="c-row"><span class="c-label">Messages</span><span class="c-value" style="color:var(--warning);">{iot_count}</span></div>
            <div class="c-row"><span class="c-label">Target</span><span class="c-value">Table Store (OTS)</span></div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="cloud-card">
            <div class="c-header">
                <span class="c-icon">📦</span>
                <span class="c-title">Alibaba Cloud OSS</span>
            </div>
            <div class="c-desc">Secure backup of anonymized high-risk (BI-RADS 4 & 5) cropped image patches. Compressed patches only — not raw DICOM.</div>
            <div class="c-row"><span class="c-label">Status</span><span class="c-value" style="color:{"#10b981" if st.session_state.oss_uploads else "#64748b"};">{"● ACTIVE" if st.session_state.oss_uploads else "○ NO UPLOADS"}</span></div>
            <div class="c-row"><span class="c-label">Bucket</span><span class="c-value">pink-edge-medical-backup</span></div>
            <div class="c-row"><span class="c-label">Region</span><span class="c-value">ap-south-1 (Mumbai)</span></div>
            <div class="c-row"><span class="c-label">Uploads</span><span class="c-value" style="color:var(--warning);">{len(st.session_state.oss_uploads)}</span></div>
        </div>
        """, unsafe_allow_html=True)

    with cc2:
        st.markdown(f"""
        <div class="cloud-card">
            <div class="c-header">
                <span class="c-icon">🔄</span>
                <span class="c-title">Alibaba Cloud ACR — OTA</span>
            </div>
        """, unsafe_allow_html=True)

        acr = st.session_state.acr_status
        if acr:
            st.markdown(f"""
            <div class="c-row"><span class="c-label">Current</span><span class="c-value">{acr['current']}</span></div>
            <div class="c-row"><span class="c-label">Available</span><span class="c-value" style="color:#10b981;">{acr['available']} ⬆</span></div>
            <div class="c-row"><span class="c-label">Size</span><span class="c-value">{acr['size']} MB</span></div>
            <div class="c-row"><span class="c-label">Registry</span><span class="c-value" style="font-size:0.7rem;">{acr['registry']}</span></div>
            """, unsafe_allow_html=True)
        else:
            st.markdown('<div style="text-align:center;padding:30px;color:var(--text-muted);font-size:0.85rem;">No OTA check performed.<br>Switch to GSM Failover and check.</div>', unsafe_allow_html=True)

        st.markdown("</div>", unsafe_allow_html=True)

        st.markdown("""
        <div class="card card-primary">
            <div class="section-header">
                <div class="s-icon">🏗️</div>
                <div class="s-title">Hybrid-Edge Architecture</div>
            </div>
            <div class="arch-box">
┌─────────────────────────────────┐
│    Rural BHU (Edge Node)        │
│   ┌─────────────────────────┐   │
│   │ RK3588 NPU (AI Inference)│   │
│   │ SQLite3 (Local Cache)   │   │
│   │ YOLOv8-OBB (INT8)       │   │
│   └─────────────────────────┘   │
│           │ 2G GSM │             │
│           ▼        ▼            │
│   ┌─────────────────────────┐   │
│   │ Alibaba Cloud IoT Hub  │   │
│   │ + OSS + ACR (OTA)      │   │
│   └─────────────────────────┘   │
└─────────────────────────────────┘
            </div>
        </div>
        """, unsafe_allow_html=True)

    st.markdown("---")
    st.markdown(f"""
    <div class="section-header">
        <div class="s-icon">💾</div>
        <div class="s-title">{t('Local Cache Status')}</div>
    </div>
    """, unsafe_allow_html=True)

    reports = get_cached_reports()
    if not reports:
        st.info(t("No reports cached"))
    else:
        table_html = f"""
        <table class="data-table">
            <thead><tr>
                <th>#</th><th>{t('Patient ID')}</th><th>{t('Modality')}</th>
                <th>BI-RADS</th><th>ACR</th><th>{t('Verdict')}</th>
                <th>{t('Timestamp')}</th><th>{t('Synced')}</th>
            </tr></thead><tbody>
        """
        for r in reports:
            sync_icon = "✅" if r[12] else "⏳"
            sync_color = "#10b981" if r[12] else "#f59e0b"
            table_html += f"""
                <tr>
                    <td>{r[0]}</td><td>{r[1]}</td><td>{r[3]}</td>
                    <td>{(r[5] or 'N/A')[:25]}</td><td>{(r[6] or 'N/A')[:25]}</td>
                    <td>{(r[7] or 'N/A')[:30]}</td><td>{r[11]}</td>
                    <td style="color:{sync_color};font-weight:600;">{sync_icon}</td>
                </tr>
            """
        table_html += "</tbody></table>"
        st.markdown(table_html, unsafe_allow_html=True)


 MAIN

In [25]:

def main():
    ensure_dirs()
    init_db()
    init_state()

    st.set_page_config(page_title="Pink Edge AI", page_icon="🩸", layout="wide", initial_sidebar_state="expanded")
    st.markdown(CSS, unsafe_allow_html=True)

    selected_model, uploaded_file = render_sidebar()

    nav = [f"🩺 {t('Dashboard')}", f"🏥 {t('Hospital Hub')}", f"☁️ {t('Cloud Sync')}"]
    st.markdown("<div style='height: 10px;'></div>", unsafe_allow_html=True)
    tabs = st.tabs(nav)

    with tabs[0]:
        render_dashboard(selected_model, uploaded_file)
    with tabs[1]:
        render_hospital_hub()
    with tabs[2]:
        render_cloud_sync()

    st.markdown(f"""
    <div class="app-footer">
        Pink Edge AI v5.3 — Clinical Intelligence Platform<br>
        Powered by Rockchip RK3588 NPU & YOLOv8-OBB (INT8 Quantized)<br>
        Alibaba Cloud AI Hackathon 2026
    </div>
    """, unsafe_allow_html=True)


if __name__ == "__main__":
    main()

2026-09-12 11:23:48.671 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 11:23:48.671 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-09-12 11:23:48.673 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 11:23:48.673 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 11:23:48.674 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-12 11:23:48.675 WARNING streamlit.runtime.scriptrunner_utils.script_run_c